# Data Preparation - Phishing Website Detector

**Projeto:** Deteção de Websites de Phishing através de Machine Learning  
**Autor:** Diogo Carvalho  
**Data:** Setembro 2026

Este notebook executa todas as etapas de preparação dos dados descritas no documento de Data Preparation, seguindo a metodologia CRISP-ML(Q).

**Etapas:**
1. Carregamento do dataset original
2. Remoção da coluna Index
3. Verificação de valores em falta
4. Verificação de duplicados
5. Análise e remoção de perfis contraditórios
6. Remoção de duplicados completos
7. Análise de outliers (diagnóstico)
8. Análise de features correlacionadas
9. Avaliação de scaling
10. Separação features/target
11. Divisão treino/teste
12. Guardar dataset preparado

## 1. Importação de bibliotecas

In [1]:
import sys
from pathlib import Path

# Adicionar a pasta src ao path
sys.path.insert(0, str(Path("..") / "src"))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from data_preparation_utils import (
    carregar_dataset,
    remover_coluna,
    verificar_valores_em_falta,
    verificar_duplicados,
    identificar_perfis_contraditorios,
    remover_perfis_contraditorios,
    remover_duplicados,
    separar_features_target,
    dividir_treino_teste,
    guardar_dataset_preparado,
    resumo_preparacao,
)

## 2. Carregamento do dataset original

In [2]:
df = carregar_dataset("phishing.csv")
df_original = df.copy()
df.head()

Dataset carregado: 11054 registos × 32 colunas


,Index,UsingIP,LongURL,ShortURL,Symbol@,Redirecting//,PrefixSuffix-,SubDomains,HTTPS,DomainRegLen,...,UsingPopupWindow,IframeRedirection,AgeofDomain,DNSRecording,WebsiteTraffic,PageRank,GoogleIndex,LinksPointingToPage,StatsReport,class
0,0,1,1,1,1,1,-1,0,1,-1,...,1,1,-1,-1,0,-1,1,1,1,-1
1,1,1,0,1,1,1,-1,-1,-1,-1,...,1,1,1,-1,1,-1,1,0,-1,-1
2,2,1,0,1,1,1,-1,-1,-1,1,...,1,1,-1,-1,1,-1,1,-1,1,-1
3,3,1,0,-1,1,1,-1,1,1,-1,...,-1,1,-1,-1,0,-1,1,1,1,1
4,4,-1,0,-1,1,-1,-1,1,1,-1,...,1,1,1,1,1,-1,1,-1,-1,1


## 3. Remoção da coluna Index

In [3]:
df = remover_coluna(df, "Index")
df.head()

Coluna 'Index' removida. Novas dimensões: (11054, 31)


,UsingIP,LongURL,ShortURL,Symbol@,Redirecting//,PrefixSuffix-,SubDomains,HTTPS,DomainRegLen,Favicon,...,UsingPopupWindow,IframeRedirection,AgeofDomain,DNSRecording,WebsiteTraffic,PageRank,GoogleIndex,LinksPointingToPage,StatsReport,class
0,1,1,1,1,1,-1,0,1,-1,1,...,1,1,-1,-1,0,-1,1,1,1,-1
1,1,0,1,1,1,-1,-1,-1,-1,1,...,1,1,1,-1,1,-1,1,0,-1,-1
2,1,0,1,1,1,-1,-1,-1,1,1,...,1,1,-1,-1,1,-1,1,-1,1,-1
3,1,0,-1,1,1,-1,1,1,-1,1,...,-1,1,-1,-1,0,-1,1,1,1,1
4,-1,0,-1,1,-1,-1,1,1,-1,1,...,1,1,1,1,1,-1,1,-1,-1,1


## 4. Verificação de valores em falta

In [4]:
verificar_valores_em_falta(df)

Total de valores em falta: 0


np.int64(0)

## 5. Verificação de duplicados

Após remover a coluna Index, surgem duplicados porque o identificador tornava registos únicos.

In [5]:
verificar_duplicados(df)

Registos duplicados: 5205


np.int64(5205)

## 6. Análise de perfis contraditórios

Perfis (combinações de features) associados a rótulos diferentes (legítimo e phishing).

In [6]:
n_perfis, n_registos, mascara = identificar_perfis_contraditorios(df, target="class")

Perfis com rótulos contraditórios: 64
Registos envolvidos em perfis contraditórios: 357


## 7. Remoção de perfis contraditórios

In [7]:
df = remover_perfis_contraditorios(df, target="class")
df.shape

Perfis com rótulos contraditórios: 64
Registos envolvidos em perfis contraditórios: 357
Registos removidos: 357
Dimensão após remoção: (10697, 31)


(10697, 31)

## 8. Remoção de duplicados completos

In [8]:
df = remover_duplicados(df)
df.shape

Duplicados removidos: 4976
Dimensão após remoção: (5721, 31)


(5721, 31)

## 9. Verificação de qualidade após limpeza

In [9]:
verificar_valores_em_falta(df)
verificar_duplicados(df)

Total de valores em falta: 0
Registos duplicados: 0


np.int64(0)

## 10. Análise de outliers (IQR) - Apenas diagnóstico

As features são categóricas ordinais (-1, 0, 1). Os "outliers" identificados pelo IQR correspondem simplesmente aos valores menos frequentes de cada feature e **não devem ser removidos**, pois representam características legítimas dos websites.

In [10]:
def analisar_outliers_iqr(df, target="class"):
    """Identifica outliers usando o método IQR."""
    features = df.drop(columns=[target])
    print(f"{'Feature':<25} {'Outliers IQR'}")
    print("-" * 45)
    total = 0
    for coluna in features.columns:
        Q1 = df[coluna].quantile(0.25)
        Q3 = df[coluna].quantile(0.75)
        IQR = Q3 - Q1
        lim_inf = Q1 - 1.5 * IQR
        lim_sup = Q3 + 1.5 * IQR
        n = ((df[coluna] < lim_inf) | (df[coluna] > lim_sup)).sum()
        if n > 0:
            print(f"{coluna:<25} {n}")
            total += n
    print("-" * 45)
    print(f"{'TOTAL':<25} {total}")
    print("\nDecisão: NENHUMA remoção (valores são categorias legítimas)")

analisar_outliers_iqr(df)

Feature                   Outliers IQR
---------------------------------------------
LongURL                   1163
ShortURL                  814
Symbol@                   1199
Redirecting//             820
PrefixSuffix-             701
Favicon                   1223
NonStdPort                943
HTTPSDomainURL            1009
InfoEmail                 1243
AbnormalURL               937
WebsiteForwarding         767
StatusBarCust             810
DisableRightClick         287
UsingPopupWindow          1275
IframeRedirection         627
GoogleIndex               985
StatsReport               884
---------------------------------------------
TOTAL                     15687

Decisão: NENHUMA remoção (valores são categorias legítimas)


## 11. Features altamente correlacionadas

Identificação de pares com |correlação de Pearson| >= 0,80.

In [11]:
corr = df.drop(columns=["class"]).corr(method="pearson").abs()
pares = []
for i in range(len(corr.columns)):
    for j in range(i):
        valor = corr.iloc[i, j]
        if valor >= 0.80:
            pares.append((corr.columns[i], corr.columns[j], round(valor, 4)))

print(f"{'Feature 1':<25} {'Feature 2':<25} {'Correlação'}")
print("-" * 70)
for f1, f2, v in pares:
    print(f"{f1:<25} {f2:<25} {v}")
print(f"\nDecisão: Variáveis mantidas. Remoção será avaliada com base no desempenho dos modelos.")

Feature 1                 Feature 2                 Correlação
----------------------------------------------------------------------
Redirecting//             ShortURL                  0.8144
NonStdPort                Favicon                   0.8118
UsingPopupWindow          Favicon                   0.9327

Decisão: Variáveis mantidas. Remoção será avaliada com base no desempenho dos modelos.


## 12. Avaliação de scaling

In [12]:
features = df.drop(columns=["class"])
print(f"Valor mínimo global: {features.min().min()}")
print(f"Valor máximo global: {features.max().max()}")
print("\nTodas as features estão no intervalo [-1, 0, 1].")
print("Decisão: Scaling NÃO aplicado (escala já uniforme).")

Valor mínimo global: -1
Valor máximo global: 1

Todas as features estão no intervalo [-1, 0, 1].
Decisão: Scaling NÃO aplicado (escala já uniforme).


## 13. Guardar dataset preparado

In [13]:
guardar_dataset_preparado(df, "phishing_prepared.csv")

Dataset preparado guardado em: /home/claude/work/diogo/notebooks/../dataset/phishing_prepared.csv
Dimensão final: (5721, 31)


## 14. Separação entre features (X) e target (y)

In [14]:
X, y = separar_features_target(df, target="class")
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

X: (5721, 30)
y: (5721,)
X shape: (5721, 30)
y shape: (5721,)


## 15. Divisão estratificada treino/teste (80/20)

In [15]:
X_train, X_test, y_train, y_test = dividir_treino_teste(X, y, test_size=0.2, random_state=42)

Treino: X_train=(4576, 30), y_train=(4576,)
Teste:  X_test=(1145, 30), y_test=(1145,)

Distribuição treino:
class
-1    51.66
 1    48.34
Name: proportion, dtype: float64

Distribuição teste:
class
-1    51.62
 1    48.38
Name: proportion, dtype: float64


## 16. Resumo final da Data Preparation

In [16]:
resumo_preparacao(df_original, df, target="class")

RESUMO DA DATA PREPARATION

Dataset original: 11054 registos × 32 colunas
Dataset preparado: 5721 registos × 31 colunas
Registos removidos: 5333

Distribuição original:
class
 1    6157
-1    4897
Name: count, dtype: int64

Distribuição preparada:
class
-1    2955
 1    2766
Name: count, dtype: int64

Valores em falta: 0
Duplicados: 0


## 17. Conclusão

A Data Preparation permitiu transformar o dataset original num conjunto adequado ao desenvolvimento de modelos de Machine Learning.

| Etapa | Resultado |
|-------|-----------|
| Dataset inicial | 11.054 registos × 32 colunas |
| Após remoção do Index | 11.054 × 31 |
| Perfis contraditórios removidos | 64 perfis (357 registos) |
| Após remoção de duplicados | 5.721 × 31 |
| Features X | 5.721 × 30 |
| Target y | 5.721 |
| Treino | 4.576 registos (80%) |
| Teste | 1.145 registos (20%) |
| Valores em falta | 0 |
| Duplicados restantes | 0 |
| Scaling | Não aplicado |
| Feature engineering | Não criadas novas features |
| Balanceamento | Não necessário (48,35% / 51,65%) |

**Próximo passo:** construção, treino e comparação de modelos (Regressão Logística, Decision Tree, Random Forest).